
---

# 💡 深度学习归一化技术笔记

## 一、 核心概念对比：LayerNorm vs. RMSNorm

| 特性 | **LayerNorm (传统)** | **RMSNorm (现代主流)** |
| :--- | :--- | :--- |
| **计算逻辑** | 减去均值（中心化）+ 除以标准差（缩放） | **只除以均方根（仅缩放）** |
| **公式核心** | $\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$ | $\hat{x}_i = \frac{x_i}{RMS(x) + \epsilon}$ |
| **可学习参数** | $\gamma$ (增益) 和 $\beta$ (偏置) | **仅 $\gamma$ (增益)** |
| **设计哲学** | 追求完整的分布标准化（0均值1方差） | 认为**缩放不变性**是关键，舍弃平移以换取效率 |
| **应用案例** | BERT, GPT-2 | **Llama 2/3, Gemma, Mistral** |

---

## 二、 深度问答记录

### Q1：LayerNorm 最后一步的 $\gamma$ 和 $\beta$ 为什么存在？
**关键点：恢复表达能力。**
* **打破约束**：归一化强行将数据限制在标准分布，可能抹杀了模型辛苦学到的特征。
* **激活函数配合**：确保输入值落在激活函数（如 Tanh, Sigmoid）的非线性区，防止模型退化为纯线性。Sigmoid/Tanh 饱和区问题：对于 Tanh 函数，0 附近几乎是线性的。如果数据全部被挤在 0 附近，多层神经网络就会退化成一个巨大的线性模型，失去处理复杂逻辑的能力。
* **提供“后悔药”**：如果模型发现不归一化效果更好，可以通过学习让 $\gamma \approx \sigma, \beta \approx \mu$ 来抵消归一化效果。

### Q2：$\gamma$ 和 $\beta$ 的操作等效于 `nn.Linear` 吗？
**关键点：不等效。它们是 Linear 的一个“极简特例”。**
* **LayerNorm ($\gamma, \beta$)**：是**逐元素（Element-wise）**的操作。每个维度独立缩放，维度间**没有信息交换**。参数量为 $2d$。
* **nn.Linear**：是**全连接（Matrix Multiplication）**的操作。输出是输入的加权和，维度间**深度融合**。参数量为 $d^2+d$。
* **结论**：LayerNorm 的这一步更像是一个“对角矩阵”变换，为了在不增加计算负担的前提下微调每个特征的强度。

---

## 三、 为什么大模型（LLM）偏爱 RMSNorm？

1.  **计算更精简**：无需计算均值，减少了大约 40% 的归一化计算开销。
2.  **收敛更稳定**：在超大规模参数下，去除均值偏移（Re-centering）并不会损害性能，反而因为简化了梯度传播路径而变得更鲁棒。
3.  **效果等价**：实验证明 RMSNorm 在 Transformer 架构中能达到与 LayerNorm 相当甚至更好的效果。

---

既然你提到了**标准公式**，我们从数学定义的角度精确拆解一下。RMSNorm 的核心逻辑是：**不进行平移（Centering），只进行缩放（Scaling）**。

---

### 1. RMSNorm 核心数学公式

假设输入的特征向量为 $x = (x_1, x_2, \dots, x_d)$，其中 $d$ 是隐藏层的维度。

#### 第一步：计算均方根 (Root Mean Square)
首先计算输入向量各个分量平方和的平均值的平方根：
$$\text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2 + \epsilon}$$
* 这里 $\epsilon$ 是为了防止分母为零的极小值。

#### 第二步：归一化 (Normalization)
将输入向量的每个分量除以这个均方根，使得归一化后的向量在单位球面上：
$$\bar{x}_i = \frac{x_i}{\text{RMS}(x)}$$

#### 第三步：再缩放 (Re-scaling)
引入一个可学习的参数 $\gamma$（代码中的 `self.weight`），对归一化后的结果进行缩放，以恢复模型的表达能力：
$$y_i = \bar{x}_i \cdot \gamma_i$$

---

### 2. 与 LayerNorm 的公式对比

为了让你看清为什么 RMSNorm 更“轻量”，对比一下 **LayerNorm** 的三个步骤：

| 步骤 | LayerNorm (标准层归一化) | RMSNorm (均方根归一化) |
| :--- | :--- | :--- |
| **1. 计算统计量** | 均值 $\mu$ **和** 方差 $\sigma^2$ | **仅** 均方根 $\text{RMS}$ |
| **2. 归一化** | $\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$ | $\bar{x}_i = \frac{x_i}{\text{RMS}(x)}$ |
| **3. 仿射变换** | $y_i = \hat{x}_i \cdot \gamma + \beta$ | $y_i = \bar{x}_i \cdot \gamma$ |



### 3. 为什么这个公式有效？
研究表明（由 Zhang 和 Sennrich 在 2019 年提出），LayerNorm 的成功主要源于其**重缩放（Re-scaling）不变性**，而与**平移（Centering）不变性**关系不大。RMSNorm 砍掉了计算均值的步骤，不仅减少了约 $10\% \sim 40\%$ 的同步开销（在 GPU 上），还在 Llama 等模型中证明了其训练稳定性完全不输给传统的 LayerNorm。


In [ ]:
import torch
import torch.nn as nn


class RMSNorm(nn.Module):
    def __init__(self,d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self,x:torch.Tensor):
        original_type = x.dtype
        rms = torch.rsqrt(x.float().pow(2).mean(-1,keepdim=True) + self.eps)
        return ((x.float() * rms) * self.weight).to(original_type)

In [4]:
def test_modules():
    # 模拟超参数
    batch_size = 2
    seq_len = 10
    d_model = 256
    print(f"=== 初始化测试参数 ===")
    print(f"Batch Size: {batch_size}, Seq Len: {seq_len}, D_Model: {d_model}")

    # 构造随机输入
    # 形状: (batch_size, seq_len, d_model)
    x = torch.randn(batch_size, seq_len, d_model)
    print(f"输入张量形状: {x.shape}")

    # ----------------------------
    # 测试 RMSNorm
    # ----------------------------
    rmsnorm = RMSNorm(d_model=d_model)
    norm_out = rmsnorm(x)
    print(f"RMSNorm 输出张量形状: {norm_out.shape}")
    assert norm_out.shape == x.shape, "RMSNorm 输出形状错误！"


test_modules()

=== 初始化测试参数 ===
Batch Size: 2, Seq Len: 10, D_Model: 256
输入张量形状: torch.Size([2, 10, 256])
RMSNorm 输出张量形状: torch.Size([2, 10, 256])
